# 07 — Model Verification and Validation

**TFM: Heterogeneity in Geopolitical Risk Shocks, Energy Prices and Macro-Financial Transmission in the Euro Area**

This notebook validates the numerical implementation used in the empirical BVAR analysis. It is self-contained and uses synthetic data generated from a known stable VAR(2).

The checks are:
1. **Known-truth recovery** of covariance and dynamic parameters.
2. **Covariance-prior scale sensitivity**: baseline $S=0$ versus $S=I_N$.
3. **Monte Carlo interval coverage** for the structural GPR-to-Gas response at impact and one year.

COVID scaling is neutralised with `theta=(1,1,1,0)`, producing a unit scale path throughout the synthetic sample.


## A. Production estimation and IRF routines

The functions below reproduce the numerical routines used by the empirical BVAR. No Drive mount or project data are required.


In [ ]:
import numpy as np
import pandas as pd

REPS, BURN, SEED = 10000, 5000, 42

def build_truncated_covid_scale_path(quarters, s0, s1, s2, rho):
    """Identical to the function used in NB04/NB05/NB06 -- must not diverge."""
    periods = pd.PeriodIndex(pd.Series(quarters).astype(str), freq="Q")
    path = np.ones(len(periods), dtype=float)
    q0, q1, q2 = pd.Period("2020Q1", "Q"), pd.Period("2020Q2", "Q"), pd.Period("2020Q3", "Q")
    trunc = pd.Period("2021Q4", "Q")
    for idx, q in enumerate(periods):
        if q == q0:
            path[idx] = s0
        elif q == q1:
            path[idx] = s1
        elif q == q2:
            path[idx] = s2
        elif q > q2 and q < trunc:
            j = q.ordinal - q2.ordinal
            path[idx] = 1.0 + (s2 - 1.0) * (rho ** j)
    return path

def estimate_var_ols(Y, p):
    """OLS VAR(p). X = [const, lag1(all vars), lag2(all vars), ...]. Homoskedastic, no COVID scaling."""
    T, n = Y.shape
    X = np.column_stack([np.ones(T - p)] + [Y[p - i - 1: T - i - 1] for i in range(p)])
    Ydep = Y[p:]
    Beta = np.linalg.lstsq(X, Ydep, rcond=None)[0]
    Uhat = Ydep - X @ Beta
    Sigma = (Uhat.T @ Uhat) / (T - p)
    return Beta, Uhat, Sigma, X, Ydep

def estimate_ar1(x):
    """Univariate AR(1) coefficient via OLS. Used for the GPR prior mean."""
    y = x[1:]; x_lag = x[:-1]
    X = np.column_stack([np.ones(len(x_lag)), x_lag])
    return np.linalg.lstsq(X, y, rcond=None)[0][1]

def build_minnesota_prior_mixed(s, n, p, gpr_idx, phi_gpr,
                                 lam1=0.2, lam2=0.5, lam3=1.0, lam4=100.0):
    """Standard mixed prior mean: GPR own-lag1 = phi_gpr (AR(1) estimated), all
    other own-lag1 means = 0 (white noise, appropriate for the already-stationary
    log-differenced/bounded variables in the baseline specification)."""
    k = 1 + n * p
    B0 = np.zeros(n * k)
    H_diag = np.zeros(n * k)
    for i in range(n):
        base = i * k
        H_diag[base] = (s[i] * lam4) ** 2
        for l in range(1, p + 1):
            for j in range(n):
                col = base + 1 + (l - 1) * n + j
                if j == i:
                    H_diag[col] = (lam1 / (l ** lam3)) ** 2
                    if l == 1 and i == gpr_idx:
                        B0[col] = phi_gpr
                else:
                    H_diag[col] = ((s[i] * lam1 * lam2) / (s[j] * (l ** lam3))) ** 2
    return B0, np.diag(H_diag)

def IWPQ(v, ixpx, rng):
    """Draw from Inverse-Wishart(v, ixpx^{-1}). Constructs a Wishart(v, ixpx) draw
    as the sum of v outer products of independent Gaussian vectors with covariance
    ixpx, generated via the Cholesky factor of ixpx (direct Gaussian construction,
    not a Bartlett decomposition), then inverts to obtain the Inverse-Wishart draw.
    Uses ONLY the passed rng -- no calls to the NumPy global random state."""
    k = ixpx.shape[0]
    L = np.linalg.cholesky(ixpx)
    A = np.zeros((v, k))
    for i in range(v):
        A[i] = L @ rng.standard_normal(k)
    return np.linalg.inv(A.T @ A)


def bvar_minnesota_gibbs_final(Y, quarters, p, gpr_idx, theta,
                                S_matrix=None, own_lag1_means=None, exclude_quarters=None,
                                reps=REPS, burn=BURN, seed=SEED,
                                lam1=0.2, lam2=0.5, lam3=1.0, lam4=100.0):
    """
    Shared production sampler, copied verbatim from NB06.
    n is inferred from Y.shape[1], so the SAME function serves the full
    6-variable system, the 4-variable reduced system, reordered/relevelled
    variants, and crisis-exclusion diagnostics.

    Prior structure: beta ~ N(B0, H), independent of Sigma a priori.
    p(Sigma) ~ |Sigma|^{-(N+1)/2} when S_matrix=None (S=0), the improper
    Jeffreys-type prior used as the NB05 baseline. alpha is FIXED at 0.
    The Sigma-prior robustness in NB06 Block C changes S_matrix only.

    theta = (s0, s1, s2, rho): frozen COVID volatility parameters estimated
    once in NB04. In THIS notebook, theta=(1,1,1,0) is used throughout,
    producing a scale path identically equal to 1 (verified directly) --
    the synthetic checks exercise the exact production code path without
    any COVID-driven volatility scaling.

    own_lag1_means: optional array of length n for alternative own-lag-1
    prior means. GPR's own-lag-1 mean is always enforced to its estimated
    AR(1) coefficient. Unused in this notebook (kept for signature parity).

    exclude_quarters: optional iterable of quarters to exclude from the
    likelihood, preserving the original quarterly calendar so that no
    artificial lag jump is introduced. Unused in this notebook (kept for
    signature parity with the production sampler).

    Sampler: reps total Gibbs iterations, burn discarded, no thinning.
    """
    rng = np.random.default_rng(seed)
    T, n = Y.shape
    q = np.asarray(quarters).astype(str)

    X_raw = np.column_stack([np.ones(T - p)] + [Y[p - i - 1:T - i - 1] for i in range(p)])
    Ydep_raw = Y[p:]

    valid = np.ones(T - p, dtype=bool)

    if exclude_quarters is not None:
        excluded = set(str(x) for x in exclude_quarters)
        for lag in range(p + 1):
            valid &= ~np.isin(q[p - lag:T - lag], list(excluded))
        if valid.sum() <= 0:
            raise ValueError("No valid VAR observations remain after applying exclude_quarters.")
        X_prior = X_raw[valid]
        Ydep_prior = Ydep_raw[valid]
        Beta_prior = np.linalg.lstsq(X_prior, Ydep_prior, rcond=None)[0]
        U_prior = Ydep_prior - X_prior @ Beta_prior
        Sigma_baseline = U_prior.T @ U_prior / valid.sum()
        ar_valid = (~np.isin(q[1:], list(excluded))) & (~np.isin(q[:-1], list(excluded)))
        X_ar = np.column_stack([np.ones(ar_valid.sum()), Y[:-1, gpr_idx][ar_valid]])
        y_ar = Y[1:, gpr_idx][ar_valid]
        phi_gpr = np.linalg.lstsq(X_ar, y_ar, rcond=None)[0][1]
    else:
        _, _, Sigma_baseline, _, _ = estimate_var_ols(Y, p)
        phi_gpr = estimate_ar1(Y[:, gpr_idx])

    s_prior = np.sqrt(np.diag(Sigma_baseline))

    s0, s1, s2, rho = theta
    full_scale = build_truncated_covid_scale_path(quarters, s0, s1, s2, rho)
    s_eff = full_scale[p:]

    X = X_raw[valid] / s_eff[valid, None]
    Ydep = Ydep_raw[valid] / s_eff[valid, None]
    T_eff = Ydep.shape[0]

    Beta_ols = np.linalg.lstsq(X, Ydep, rcond=None)[0]

    if own_lag1_means is None:
        B0, H = build_minnesota_prior_mixed(s_prior, n, p, gpr_idx, phi_gpr, lam1, lam2, lam3, lam4)
    else:
        means = np.asarray(own_lag1_means, dtype=float).copy()
        if len(means) != n:
            raise ValueError("own_lag1_means must have length n.")
        means[gpr_idx] = phi_gpr
        B0, H = build_minnesota_prior_mixed_general(s_prior, n, p, gpr_idx, means,
                                                     lam1, lam2, lam3, lam4)

    Hinv = np.linalg.inv(H)

    alpha = 0
    S = np.zeros((n, n)) if S_matrix is None else S_matrix

    betaols_vec = Beta_ols.T.reshape(-1)
    Sigma = (Ydep - X @ Beta_ols).T @ (Ydep - X @ Beta_ols) / T_eff

    k = 1 + n * p
    XtX = X.T @ X

    beta_draws = np.zeros((reps - burn, n, k))
    sigma_draws = np.zeros((reps - burn, n, n))

    for it in range(reps):
        SigInv = np.linalg.inv(Sigma)
        V = np.linalg.inv(Hinv + np.kron(SigInv, XtX))
        M = V @ (Hinv @ B0 + np.kron(SigInv, XtX) @ betaols_vec)

        beta_vec = M + np.linalg.cholesky(V) @ rng.standard_normal(n * k)
        Beta = beta_vec.reshape(n, k).T

        Uhat_it = Ydep - X @ Beta
        scale = Uhat_it.T @ Uhat_it + S
        Sigma = IWPQ(T_eff + alpha, np.linalg.inv(scale), rng)

        if it >= burn:
            beta_draws[it - burn] = Beta.T
            sigma_draws[it - burn] = Sigma

    return beta_draws, sigma_draws, phi_gpr


def build_minnesota_prior_mixed_general(s, n, p, gpr_idx, own_lag1_means,
                                         lam1=0.2, lam2=0.5, lam3=1.0, lam4=100.0):
    """Same as build_minnesota_prior_mixed, but the own-lag1 prior mean of
    every variable is supplied explicitly via own_lag1_means (length n),
    instead of hardcoding 0 for all non-GPR variables. Unused in this
    notebook (kept for signature parity with NB06)."""
    k = 1 + n * p
    B0 = np.zeros(n * k)
    H_diag = np.zeros(n * k)
    for i in range(n):
        base = i * k
        H_diag[base] = (s[i] * lam4) ** 2
        for l in range(1, p + 1):
            for j in range(n):
                col = base + 1 + (l - 1) * n + j
                if j == i:
                    H_diag[col] = (lam1 / (l ** lam3)) ** 2
                    if l == 1:
                        B0[col] = own_lag1_means[i]
                else:
                    H_diag[col] = ((s[i] * lam1 * lam2) / (s[j] * (l ** lam3))) ** 2
    return B0, np.diag(H_diag)


def compute_structural_irfs(Phis, Sigma, horizon, n, p):
    Psi = [np.eye(n)]
    for j in range(1, horizon + 1):
        psi_j = np.zeros((n, n))
        for i in range(1, min(j, p) + 1):
            psi_j += Phis[i - 1] @ Psi[j - i]
        Psi.append(psi_j)
    Q = np.linalg.cholesky(Sigma)
    return np.array([psi @ Q for psi in Psi])

def phis_from_beta(Beta_T, n, p):
    return [Beta_T[:, 1 + i*n : 1 + (i+1)*n] for i in range(p)]

def bvar_structural_irfs(beta_draws, sigma_draws, n, p, horizon):
    n_draws = beta_draws.shape[0]
    Theta_draws = np.zeros((n_draws, horizon + 1, n, n))
    for d in range(n_draws):
        Phis = phis_from_beta(beta_draws[d], n, p)
        Theta_draws[d] = compute_structural_irfs(Phis, sigma_draws[d], horizon, n, p)
    return Theta_draws

def bootstrap_structural_irfs(Y, p, horizon, B, seed=None):
    """Classical bootstrap -- unchanged, no Sigma prior involved."""
    rng = np.random.default_rng(seed)
    T, n = Y.shape
    Beta, Uhat, Sigma, X, Ydep = estimate_var_ols(Y, p)
    Phis = [Beta[1+i*n:1+(i+1)*n, :].T for i in range(p)]
    Theta_point = compute_structural_irfs(Phis, Sigma, horizon, n, p)
    Theta_boot = np.zeros((B, horizon + 1, n, n))
    for b in range(B):
        Ystar = np.zeros((T, n)); Ystar[:p] = Y[:p]
        for t in range(p, T):
            idx = rng.integers(0, Uhat.shape[0])
            lags = np.concatenate([Ystar[t - i - 1] for i in range(p)])
            xt = np.concatenate([[1], lags])
            Ystar[t] = xt @ Beta + Uhat[idx]
        Beta_b, Uhat_b, Sigma_b, Xb, Ydepb = estimate_var_ols(Ystar, p)
        Phis_b = [Beta_b[1+i*n:1+(i+1)*n, :].T for i in range(p)]
        Theta_boot[b] = compute_structural_irfs(Phis_b, Sigma_b, horizon, n, p)
    return Theta_point, Theta_boot

NEUTRAL_THETA = (1, 1, 1, 0)  # s0=s1=s2=1, rho=0 -> scale path identically 1 (verified)

print("Functions loaded (verbatim copy of NB06's final production sampler).")

Functions loaded (verbatim copy of NB06's final production sampler).


## B. Synthetic data-generating process

A stable six-variable VAR(2) is simulated with the same dimensionality, lag order and sample length as the empirical systems. Residual scales are deliberately heterogeneous, making the covariance-prior scale check informative.


In [ ]:
n, p, T = 6, 2, 115  # 115 observations -> 113 effective observations with p=2

true_A1 = np.diag([0.85, 0.2, 0.2, 0.15, 0.15, 0.2])  # GPR (idx 0) most persistent
true_A1[2, 0] = 0.05  # GPR -> Gas, small dynamic link

true_A2 = 0.1 * np.eye(n)

# Verify stability of the synthetic VAR(2)
companion = np.block([
    [true_A1, true_A2],
    [np.eye(n), np.zeros((n, n))]
])

spectral_radius = np.max(np.abs(np.linalg.eigvals(companion)))
assert spectral_radius < 1, "Synthetic VAR is not stable."

true_sigma_diag = np.array([
    0.15,   # GPR
    0.10,   # Brent
    0.15,   # Gas
    0.006,  # HICP
    0.015,  # IPI
    0.02    # CLIFS
])

true_Sigma = np.diag(true_sigma_diag**2)

# Contemporaneous GPR-Gas correlation
true_Sigma[0, 2] = true_Sigma[2, 0] = (
    0.3 * true_sigma_diag[0] * true_sigma_diag[2]
)

quarters = pd.period_range(
    "1997Q1",
    periods=T,
    freq="Q"
).astype(str)

SIM_BURN = 300


def simulate(seed):
    rng_local = np.random.default_rng(seed)

    Y_full = np.zeros((T + SIM_BURN, n))
    L = np.linalg.cholesky(true_Sigma)

    for t in range(2, T + SIM_BURN):
        Y_full[t] = (
            Y_full[t - 1] @ true_A1.T
            + Y_full[t - 2] @ true_A2.T
            + L @ rng_local.standard_normal(n)
        )

    return Y_full[SIM_BURN:]


Y = simulate(seed=0)

print(f"Companion spectral radius: {spectral_radius:.4f}")
print("Synthetic data generated:", Y.shape)
print("Quarters:", quarters[0], "to", quarters[-1])

Companion spectral radius: 0.9547
Synthetic data generated: (115, 6)
Quarters: 1997Q1 to 2025Q3


## C. Check 1 — Known-truth recovery

The production BVAR sampler is applied to the synthetic sample with neutral COVID scaling. Covariance recovery should be close to the known DGP values. Dynamic coefficients are expected to exhibit Minnesota shrinkage, so exact coefficient recovery is not required.


In [ ]:
beta_draws, sigma_draws, phi_gpr = bvar_minnesota_gibbs_final(
    Y, quarters,
    p=p,
    gpr_idx=0,
    theta=NEUTRAL_THETA,
    reps=REPS,
    burn=BURN,
    seed=1
)

# ============================================================
# CHECK 1. RECOVERY OF KNOWN DGP PARAMETERS
# ============================================================

print("=== Sigma recovery (diffuse Jeffreys-type prior) ===")

sqrt_post_mean_variance = np.sqrt(
    np.diagonal(sigma_draws, axis1=1, axis2=2).mean(axis=0)
)

print("True residual std:          ", true_sigma_diag.round(4))
print("Sqrt posterior mean var:    ", sqrt_post_mean_variance.round(4))
print(
    "Max relative error:         ",
    (
        np.abs(sqrt_post_mean_variance - true_sigma_diag)
        / true_sigma_diag
    ).max().round(3)
)

print("\n=== B own-lag1 recovery (expected Minnesota shrinkage) ===")

post_mean_own_lag1 = np.diag(
    beta_draws[:, :, 1:1+n].mean(axis=0)
)

print("True own-lag1 diagonal: ", np.diag(true_A1).round(3))
print("Posterior mean:         ", post_mean_own_lag1.round(3))
print(
    "GPR AR(1) estimated (prior mean):",
    round(phi_gpr, 3)
)

=== Sigma recovery (diffuse Jeffreys-type prior) ===
True residual std:           [0.15  0.1   0.15  0.006 0.015 0.02 ]
Sqrt posterior mean var:     [0.1641 0.0942 0.1532 0.006  0.0155 0.0222]
Max relative error:          0.108

=== B own-lag1 recovery (expected Minnesota shrinkage) ===
True own-lag1 diagonal:  [0.85 0.2  0.2  0.15 0.15 0.2 ]
Posterior mean:          [0.78  0.102 0.073 0.064 0.097 0.103]
GPR AR(1) estimated (prior mean): 0.85


## D. Check 2 — Covariance-prior scale sensitivity

The same synthetic sample is estimated twice, changing only the covariance-prior scale matrix: baseline $S=0$ versus $S=I_N$. This isolates the scale effect of the covariance prior.


In [ ]:
# ============================================================
# CHECK 2. COVARIANCE-PRIOR SCALE SENSITIVITY
# ============================================================

_, sigma_draws_S0, _ = bvar_minnesota_gibbs_final(
    Y, quarters,
    p=p,
    gpr_idx=0,
    theta=NEUTRAL_THETA,
    S_matrix=None,
    reps=REPS,
    burn=BURN,
    seed=1
)

_, sigma_draws_SI, _ = bvar_minnesota_gibbs_final(
    Y, quarters,
    p=p,
    gpr_idx=0,
    theta=NEUTRAL_THETA,
    S_matrix=np.eye(n),
    reps=REPS,
    burn=BURN,
    seed=1
)

sqrt_post_mean_var_S0 = np.sqrt(
    np.diagonal(sigma_draws_S0, axis1=1, axis2=2).mean(axis=0)
)

sqrt_post_mean_var_SI = np.sqrt(
    np.diagonal(sigma_draws_SI, axis1=1, axis2=2).mean(axis=0)
)

print("=== Covariance-prior scale sensitivity ===")

print("True residual std:          ", true_sigma_diag.round(4))

print("\nS=0:")
print("Sqrt posterior mean var:    ", sqrt_post_mean_var_S0.round(4))
print(
    "Scale ratio:                ",
    (sqrt_post_mean_var_S0 / true_sigma_diag).round(2)
)

print("\nS=I:")
print("Sqrt posterior mean var:    ", sqrt_post_mean_var_SI.round(4))
print(
    "Scale ratio:                ",
    (sqrt_post_mean_var_SI / true_sigma_diag).round(2)
)

=== Covariance-prior scale sensitivity ===
True residual std:           [0.15  0.1   0.15  0.006 0.015 0.02 ]

S=0:
Sqrt posterior mean var:     [0.1641 0.0942 0.1532 0.006  0.0155 0.0222]
Scale ratio:                 [1.09 0.94 1.02 0.99 1.03 1.11]

S=I:
Sqrt posterior mean var:     [0.1919 0.1368 0.1829 0.0978 0.099  0.1004]
Scale ratio:                 [ 1.28  1.37  1.22 16.3   6.6   5.02]


## E. Check 3 — Monte Carlo interval coverage

One hundred independent samples are simulated from the known DGP. For each sample, 90% intervals are constructed for the structural GPR-to-Gas response at $h=0$ and $h=4$. A classical residual-bootstrap interval is retained only as a numerical validation benchmark; it is not an empirical VAR specification in the public analysis.


In [ ]:
# ============================================================
# CHECK 3. MONTE CARLO COVERAGE
#
# Coverage is evaluated for the GPR -> Gas structural response
# at h=0 and h=4.
#
# h=0 primarily checks contemporaneous covariance/Cholesky
# recovery, while h=4 additionally checks VAR dynamics.
# ============================================================

MC_HORIZON = 4
N_MC = 100
N_BOOT = 500

true_Theta = compute_structural_irfs(
    [true_A1, true_A2],
    true_Sigma,
    MC_HORIZON,
    n,
    p
)

true_irf = {
    0: true_Theta[0, 2, 0],
    4: true_Theta[4, 2, 0],
}

print("True GPR -> Gas responses:")
for h, value in true_irf.items():
    print(f"h={h}: {value:.5f}")

boot_covers = {0: 0, 4: 0}
bvar_covers = {0: 0, 4: 0}

for m in range(N_MC):

    Ym = simulate(seed=1000 + m)

    # Classical bootstrap
    _, Theta_boot = bootstrap_structural_irfs(
        Ym,
        p,
        MC_HORIZON,
        B=N_BOOT,
        seed=2000 + m
    )

    # BVAR
    beta_d, sigma_d, _ = bvar_minnesota_gibbs_final(
        Ym,
        quarters,
        p=p,
        gpr_idx=0,
        theta=NEUTRAL_THETA,
        reps=REPS,
        burn=BURN,
        seed=3000 + m
    )

    Theta_bvar = bvar_structural_irfs(
        beta_d,
        sigma_d,
        n,
        p,
        MC_HORIZON
    )

    for h in [0, 4]:

        bl, bh = np.percentile(
            Theta_boot[:, h, 2, 0],
            [5, 95]
        )

        vl, vh = np.percentile(
            Theta_bvar[:, h, 2, 0],
            [5, 95]
        )

        boot_covers[h] += bl <= true_irf[h] <= bh
        bvar_covers[h] += vl <= true_irf[h] <= vh

    if (m + 1) % 10 == 0:
        print(f"...{m + 1}/{N_MC} Monte Carlo replications done")


print("\n=== Monte Carlo 90% interval coverage ===")

for h in [0, 4]:

    boot_pct = 100 * boot_covers[h] / N_MC
    bvar_pct = 100 * bvar_covers[h] / N_MC

    print(f"\nh={h}")
    print(
        f"Classical bootstrap: {boot_covers[h]}/{N_MC} "
        f"= {boot_pct:.1f}% (nominal 90%)"
    )
    print(
        f"BVAR:                {bvar_covers[h]}/{N_MC} "
        f"= {bvar_pct:.1f}% (nominal 90%)"
    )

True GPR -> Gas responses:
h=0: 0.04500
h=4: 0.00943
...10/100 Monte Carlo replications done
...20/100 Monte Carlo replications done
...30/100 Monte Carlo replications done
...40/100 Monte Carlo replications done
...50/100 Monte Carlo replications done
...60/100 Monte Carlo replications done
...70/100 Monte Carlo replications done
...80/100 Monte Carlo replications done
...90/100 Monte Carlo replications done
...100/100 Monte Carlo replications done

=== Monte Carlo 90% interval coverage ===

h=0
Classical bootstrap: 89/100 = 89.0% (nominal 90%)
BVAR:                92/100 = 92.0% (nominal 90%)

h=4
Classical bootstrap: 85/100 = 85.0% (nominal 90%)
BVAR:                90/100 = 90.0% (nominal 90%)


## F. Validation summary

These exercises check numerical behaviour rather than add empirical results. The baseline BVAR should recover covariance scale reasonably closely; $S=I_N$ should reveal scale distortion for low-volatility variables; and Monte Carlo coverage can be compared with the nominal 90% level at impact and one year.

These checks validate implementation of estimation and structural-IRF routines, not the empirical identifying assumptions themselves.
